In [1]:
import json
import numpy as np
import pandas as pd

TECHNIQUES  = ["zero_shot", "cot", "guided_cot", "tot", "got"]
CATEGORIES  = ["simple", "medium", "hard"]
ALL_METRICS = ["fcr", "scs", "wr", "roa", "orr", "wqs"]


def load_file(path):
    with open(path, "r") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    df["category"] = df["image_id"].str.split("_").str[0]
    df["img_num"]  = df["image_id"].str.split("_").str[1].astype(int)
    return df


def load_and_merge(file_1, file_2):
    df1 = load_file(file_1)
    df2 = load_file(file_2)

    r1_name = df1["evaluator"].iloc[0]
    r2_name = df2["evaluator"].iloc[0]

    print(f"\n  Reviewer 1 : {r1_name}  ({len(df1)} records)")
    print(f"  Reviewer 2 : {r2_name}  ({len(df2)} records)")

    merged = pd.merge(
        df1, df2,
        on=["image_id", "technique", "category", "img_num"],
        suffixes=(f"_{r1_name}", f"_{r2_name}")
    )

    # Averaged scores column for each metric
    for m in ALL_METRICS:
        c1, c2 = f"{m}_{r1_name}", f"{m}_{r2_name}"
        if c1 in merged.columns and c2 in merged.columns:
            merged[f"{m}_avg"] = merged[[c1, c2]].mean(axis=1)

    print(f"  Merged     : {len(merged)} records")
    print(f"  Techniques : {sorted(merged['technique'].unique())}")
    print(f"  Categories : {sorted(merged['category'].unique())}")

    return merged, r1_name, r2_name

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.metrics import cohen_kappa_score
from pingouin import intraclass_corr

ALL_METRICS    = ["fcr", "scs", "wr", "roa", "orr", "wqs"]
RUBRIC_METRICS = ["scs", "wr", "roa"]   # ordinal 0/0.25/0.5/0.75/1.0
TECHNIQUES     = ["zero_shot", "cot", "guided_cot", "tot", "got"]

TECHNIQUE_COLORS = {
    "zero_shot"  : "#4C72B0",
    "cot"        : "#DD8452",
    "guided_cot" : "#55A868",
    "tot"        : "#C44E52",
    "got"        : "#8172B2",
}


# ── Helpers ───────────────────────────────────────────────────────────────────

def cohens_kappa_safe(s1, s2):
    mask = s1.notna() & s2.notna()
    a, b = s1[mask], s2[mask]
    if len(a) < 2 or a.nunique() < 2:
        return np.nan
    vals = sorted(set(a) | set(b))
    m = {v: i for i, v in enumerate(vals)}
    return cohen_kappa_score(a.map(m), b.map(m))


def icc_safe(df, col1, col2):
    tmp = df[[col1, col2]].dropna().copy()
    if len(tmp) < 3:
        return np.nan
    long = pd.DataFrame({
        "targets" : list(range(len(tmp))) * 2,
        "raters"  : [1] * len(tmp) + [2] * len(tmp),
        "scores"  : list(tmp[col1]) + list(tmp[col2]),
    })
    try:
        res = intraclass_corr(data=long, targets="targets",
                              raters="raters", ratings="scores")
        return res.loc[res["Type"] == "ICC2", "ICC"].values[0]
    except Exception:
        return np.nan


# ── Main ──────────────────────────────────────────────────────────────────────

def run_irr(merged, r1_name, r2_name, out_dir, disagree_thresh=0.25):
    print("\n" + "─"*60)
    print("  [1/4] INTER-RATER RELIABILITY")
    print("─"*60)

    # ── Summary table ─────────────────────────────────────────────────────────
    rows = []
    for m in ALL_METRICS:
        c1, c2 = f"{m}_{r1_name}", f"{m}_{r2_name}"
        if c1 not in merged.columns or c2 not in merged.columns:
            continue
        s1, s2 = merged[c1], merged[c2]
        diff   = (s1 - s2).abs()
        kappa  = cohens_kappa_safe(s1, s2) if m in RUBRIC_METRICS else np.nan
        icc    = icc_safe(merged, c1, c2)
        rows.append({
            "Metric"            : m.upper(),
            "N"                 : int(s1.notna().sum()),
            f"Mean_{r1_name}"   : round(s1.mean(), 3),
            f"Mean_{r2_name}"   : round(s2.mean(), 3),
            "Mean|diff|"        : round(diff.mean(), 3),
            "Max|diff|"         : round(diff.max(), 3),
            "Cohen_kappa"       : round(kappa, 3) if not np.isnan(kappa) else None,
            "ICC(2,1)"          : round(icc,   3) if not np.isnan(icc)   else None,
        })

    irr_df = pd.DataFrame(rows).set_index("Metric")
    print("\n  IRR Summary:")
    print(irr_df.to_string())
    irr_df.to_csv(f"{out_dir}/irr_summary.csv")
    print(f"  → Saved: {out_dir}/irr_summary.csv")

    # ── Disagreement heatmaps ─────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for ax, m in zip(axes, ALL_METRICS):
        c1, c2 = f"{m}_{r1_name}", f"{m}_{r2_name}"
        if c1 not in merged.columns or c2 not in merged.columns:
            ax.set_visible(False)
            continue
        tmp = merged.copy()
        tmp["diff"] = (tmp[c1] - tmp[c2]).abs()
        heat = tmp.pivot_table(index="image_id", columns="technique",
                               values="diff", aggfunc="mean")
        cols = [t for t in TECHNIQUES if t in heat.columns]
        sns.heatmap(heat[cols], ax=ax, cmap="YlOrRd", vmin=0, vmax=1,
                    linewidths=0.4, annot=True, fmt=".2f",
                    annot_kws={"size": 7})
        ax.set_title(f"{m.upper()} — |diff|", fontweight="bold")
        ax.tick_params(axis="x", rotation=30)
    plt.suptitle("Inter-Rater Disagreement Heatmaps", fontsize=14,
                 fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(f"{out_dir}/irr_heatmaps.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Saved: {out_dir}/irr_heatmaps.png")

    # ── Scatter plots ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    for ax, m in zip(axes, ALL_METRICS):
        c1, c2 = f"{m}_{r1_name}", f"{m}_{r2_name}"
        if c1 not in merged.columns or c2 not in merged.columns:
            ax.set_visible(False)
            continue
        tmp = merged[["technique", c1, c2]].dropna()
        for tech, grp in tmp.groupby("technique"):
            ax.scatter(grp[c1], grp[c2], label=tech,
                       color=TECHNIQUE_COLORS.get(tech, "grey"), alpha=0.7, s=55)
        ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
        ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel(r1_name); ax.set_ylabel(r2_name)
        ax.set_title(m.upper(), fontweight="bold")
    handles = [mpatches.Patch(color=TECHNIQUE_COLORS[t], label=t)
               for t in TECHNIQUES if t in merged["technique"].unique()]
    fig.legend(handles=handles, loc="lower center", ncol=5,
               bbox_to_anchor=(0.5, -0.03), title="Technique")
    plt.suptitle(f"Reviewer Agreement: {r1_name} vs {r2_name}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/irr_scatter.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Saved: {out_dir}/irr_scatter.png")

    # ── Flagged disagreements ─────────────────────────────────────────────────
    flagged_parts = []
    for m in ALL_METRICS:
        c1, c2 = f"{m}_{r1_name}", f"{m}_{r2_name}"
        if c1 not in merged.columns or c2 not in merged.columns:
            continue
        tmp = merged[["image_id", "technique", c1, c2]].dropna().copy()
        tmp["diff"] = (tmp[c1] - tmp[c2]).abs()
        bad = tmp[tmp["diff"] >= disagree_thresh].copy()
        bad["metric"] = m.upper()
        bad.rename(columns={c1: r1_name, c2: r2_name}, inplace=True)
        flagged_parts.append(bad[["image_id", "technique", "metric",
                                   r1_name, r2_name, "diff"]])

    if flagged_parts:
        flagged = (pd.concat(flagged_parts)
                   .sort_values(["diff", "metric"], ascending=[False, True])
                   .reset_index(drop=True))
        print(f"\n  Flagged cases (|diff| >= {disagree_thresh}): {len(flagged)}")
        print(flagged.to_string())
        flagged.to_csv(f"{out_dir}/irr_flagged.csv", index=False)
        print(f"  → Saved: {out_dir}/irr_flagged.csv")
    else:
        print(f"\n  No disagreements >= {disagree_thresh} found.")

In [3]:
"""
technique.py — Technique Comparison
  - Mean WQS per technique (bar chart + stats)
  - WQS distribution (boxplot)
  - Kruskal-Wallis + pairwise Mann-Whitney U
  - Radar chart across all metrics
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations

TECHNIQUES = ["zero_shot", "cot", "guided_cot", "tot", "got"]
ALL_METRICS = ["fcr", "scs", "wr", "roa", "orr", "wqs"]

TECHNIQUE_COLORS = {
    "zero_shot"  : "#4C72B0",
    "cot"        : "#DD8452",
    "guided_cot" : "#55A868",
    "tot"        : "#C44E52",
    "got"        : "#8172B2",
}


def run_technique_analysis(merged, r1_name, r2_name, out_dir):
    print("\n" + "─"*60)
    print("  [2/4] TECHNIQUE COMPARISON")
    print("─"*60)

    order = [t for t in TECHNIQUES if t in merged["technique"].unique()]

    # ── Summary table ─────────────────────────────────────────────────────────
    summary = (
        merged.groupby("technique")["wqs_avg"]
        .agg(["mean", "std", "median", "count"])
        .reindex(order).round(4)
    )
    summary["SE"] = (summary["std"] / np.sqrt(summary["count"])).round(4)
    print("\n  WQS Summary by Technique:")
    print(summary.to_string())
    summary.to_csv(f"{out_dir}/technique_wqs_summary.csv")
    print(f"  → Saved: {out_dir}/technique_wqs_summary.csv")

    # ── Bar chart ─────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    colors = [TECHNIQUE_COLORS.get(t, "grey") for t in order]
    bars = ax.bar(order, summary["mean"], yerr=summary["SE"],
                  capsize=5, color=colors, edgecolor="white", alpha=0.9)
    for bar, v in zip(bars, summary["mean"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
                f"{v:.3f}", ha="center", va="bottom", fontweight="bold", fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Mean WQS (± SE)")
    ax.set_title("Mean WQS by Technique", fontweight="bold")
    ax.set_xlabel("Technique")

    # ── Boxplot ───────────────────────────────────────────────────────────────
    ax = axes[1]
    palette = {t: TECHNIQUE_COLORS.get(t, "grey") for t in order}
    sns.boxplot(data=merged, x="technique", y="wqs_avg", order=order,
                palette=palette, ax=ax, linewidth=1.2)
    sns.stripplot(data=merged, x="technique", y="wqs_avg", order=order,
                  color="black", alpha=0.35, size=4, ax=ax, jitter=True)
    ax.set_ylim(-0.05, 1.15)
    ax.set_ylabel("WQS (averaged reviewers)")
    ax.set_title("WQS Distribution by Technique", fontweight="bold")
    ax.set_xlabel("Technique")

    plt.tight_layout()
    plt.savefig(f"{out_dir}/technique_wqs.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Saved: {out_dir}/technique_wqs.png")

    # ── Kruskal-Wallis ────────────────────────────────────────────────────────
    groups = [merged.loc[merged["technique"] == t, "wqs_avg"].dropna().values
              for t in order]
    stat, p = stats.kruskal(*groups)
    print(f"\n  Kruskal-Wallis: H = {stat:.4f}, p = {p:.4f}")
    sig_str = "Significant" if p < 0.05 else "Not significant"
    print(f"  → {sig_str} difference between techniques (p {'<' if p < 0.05 else '>='} 0.05)")

    # ── Pairwise Mann-Whitney U ───────────────────────────────────────────────
    pw_rows = []
    print(f"\n  {'Pair':<35} {'U':>8} {'p':>10}  Sig")
    print("  " + "─"*55)
    for t1, t2 in combinations(order, 2):
        g1 = merged.loc[merged["technique"] == t1, "wqs_avg"].dropna()
        g2 = merged.loc[merged["technique"] == t2, "wqs_avg"].dropna()
        u, pval = stats.mannwhitneyu(g1, g2, alternative="two-sided")
        sig = ("***" if pval < 0.001 else
               "**"  if pval < 0.01  else
               "*"   if pval < 0.05  else "ns")
        print(f"  {t1} vs {t2:<22} {u:>8.1f} {pval:>10.4f}  {sig}")
        pw_rows.append({"pair": f"{t1} vs {t2}", "U": u, "p": pval, "sig": sig})

    pd.DataFrame(pw_rows).to_csv(f"{out_dir}/technique_pairwise.csv", index=False)
    print(f"  → Saved: {out_dir}/technique_pairwise.csv")

    # ── Reviewer side-by-side bar ─────────────────────────────────────────────
    c1_wqs = f"wqs_{r1_name}"
    c2_wqs = f"wqs_{r2_name}"
    if c1_wqs in merged.columns and c2_wqs in merged.columns:
        r1_means = merged.groupby("technique")[c1_wqs].mean().reindex(order)
        r2_means = merged.groupby("technique")[c2_wqs].mean().reindex(order)
        x = np.arange(len(order))
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(x - 0.2, r1_means, width=0.35, label=r1_name,
               color="#4C72B0", alpha=0.85)
        ax.bar(x + 0.2, r2_means, width=0.35, label=r2_name,
               color="#DD8452", alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(order)
        ax.set_ylim(0, 1.15)
        ax.set_ylabel("Mean WQS")
        ax.set_title("WQS by Technique — Reviewer Comparison", fontweight="bold")
        ax.legend(title="Reviewer")
        plt.tight_layout()
        plt.savefig(f"{out_dir}/technique_reviewer_comparison.png",
                    dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  → Saved: {out_dir}/technique_reviewer_comparison.png")

    # ── Radar chart ───────────────────────────────────────────────────────────
    radar_cols   = ["fcr_avg", "scs_avg", "wr_avg", "roa_avg", "wqs_avg"]
    radar_labels = ["FCR", "SCS", "WR", "ROA", "WQS"]
    avail = [c for c in radar_cols if c in merged.columns]
    labels = [radar_labels[radar_cols.index(c)] for c in avail]

    tech_radar = merged.groupby("technique")[avail].mean().reindex(order)
    N = len(avail)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    for tech in order:
        vals = tech_radar.loc[tech, avail].fillna(0).tolist()
        vals += vals[:1]
        color = TECHNIQUE_COLORS.get(tech, "grey")
        ax.plot(angles, vals, color=color, linewidth=2, label=tech)
        ax.fill(angles, vals, color=color, alpha=0.07)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=12)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], fontsize=8)
    ax.set_title("Technique Profiles — All Metrics", fontsize=13,
                 fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{out_dir}/technique_radar.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Saved: {out_dir}/technique_radar.png")

In [ ]:
"""
difficulty.py — Difficulty Analysis (simple / medium / hard)
  - Mean WQS per difficulty
  - WQS by technique × difficulty (grouped bar + heatmap)
  - Kruskal-Wallis across difficulty levels
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

TECHNIQUES = ["zero_shot", "cot", "guided_cot", "tot", "got"]
CATEGORIES  = ["simple", "medium", "hard"]

TECHNIQUE_COLORS = {
    "zero_shot"  : "#4C72B0",
    "cot"        : "#DD8452",
    "guided_cot" : "#55A868",
    "tot"        : "#C44E52",
    "got"        : "#8172B2",
}
CATEGORY_COLORS = {
    "simple" : "#2ecc71",
    "medium" : "#f39c12",
    "hard"   : "#e74c3c",
}


def run_difficulty_analysis(merged, out_dir):
    print("\n" + "─"*60)
    print("  [3/4] DIFFICULTY ANALYSIS")
    print("─"*60)

    cat_order  = [c for c in CATEGORIES  if c in merged["category"].unique()]
    tech_order = [t for t in TECHNIQUES  if t in merged["technique"].unique()]

    # ── Summary table ─────────────────────────────────────────────────────────
    summary = (
        merged.groupby("category")["wqs_avg"]
        .agg(["mean", "std", "median", "count"])
        .reindex(cat_order).round(4)
    )
    print("\n  WQS by Difficulty:")
    print(summary.to_string())
    summary.to_csv(f"{out_dir}/difficulty_wqs_summary.csv")
    print(f"  → Saved: {out_dir}/difficulty_wqs_summary.csv")

    # ── Kruskal-Wallis ────────────────────────────────────────────────────────
    groups = [merged.loc[merged["category"] == c, "wqs_avg"].dropna().values
              for c in cat_order]
    if len(groups) >= 2:
        stat, p = stats.kruskal(*groups)
        print(f"\n  Kruskal-Wallis (difficulty): H = {stat:.4f}, p = {p:.4f}")
        print(f"  → {'Significant' if p < 0.05 else 'Not significant'} "
              f"difference across difficulty levels")

    # ── Bar + grouped bar ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    colors = [CATEGORY_COLORS.get(c, "grey") for c in cat_order]
    bars = ax.bar(cat_order, summary["mean"], color=colors,
                  edgecolor="white", alpha=0.9)
    for bar, v in zip(bars, summary["mean"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
                f"{v:.3f}", ha="center", va="bottom", fontweight="bold")
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Mean WQS")
    ax.set_title("Mean WQS by Difficulty", fontweight="bold")

    ax = axes[1]
    pivot = (merged.groupby(["technique", "category"])["wqs_avg"]
             .mean().unstack().reindex(index=tech_order, columns=cat_order))
    x = np.arange(len(tech_order))
    w = 0.25
    for i, cat in enumerate(cat_order):
        ax.bar(x + i*w, pivot[cat], width=w, label=cat,
               color=CATEGORY_COLORS.get(cat, "grey"), alpha=0.85)
    ax.set_xticks(x + w * (len(cat_order)-1)/2)
    ax.set_xticklabels(tech_order, rotation=15)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Mean WQS")
    ax.set_title("WQS — Technique × Difficulty", fontweight="bold")
    ax.legend(title="Difficulty")

    plt.tight_layout()
    plt.savefig(f"{out_dir}/difficulty_wqs.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Saved: {out_dir}/difficulty_wqs.png")

    # ── Heatmap ───────────────────────────────────────────────────────────────
    heat = (merged.groupby(["technique", "category"])["wqs_avg"]
            .mean().unstack()
            .reindex(index=tech_order, columns=cat_order))

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.heatmap(heat, annot=True, fmt=".3f", cmap="RdYlGn",
                vmin=0, vmax=1, linewidths=0.5, ax=ax,
                cbar_kws={"label": "Mean WQS"})
    ax.set_title("Mean WQS — Technique × Difficulty", fontsize=13, fontweight="bold")
    ax.set_xlabel("Difficulty")
    ax.set_ylabel("Technique")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/difficulty_heatmap.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Saved: {out_dir}/difficulty_heatmap.png")

    # ── Boxplot per difficulty ────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 5))
    palette = {c: CATEGORY_COLORS.get(c, "grey") for c in cat_order}
    sns.boxplot(data=merged, x="category", y="wqs_avg", order=cat_order,
                palette=palette, ax=ax, linewidth=1.2)
    sns.stripplot(data=merged, x="category", y="wqs_avg", order=cat_order,
                  color="black", alpha=0.3, size=4, ax=ax, jitter=True)
    ax.set_ylim(-0.05, 1.15)
    ax.set_ylabel("WQS (averaged reviewers)")
    ax.set_xlabel("Difficulty")
    ax.set_title("WQS Distribution by Difficulty", fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/difficulty_boxplot.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Saved: {out_dir}/difficulty_boxplot.png")

In [5]:
"""
metrics.py — Per-Metric Breakdown (FCR, SCS, WR, ROA, ORR, WQS)
  - Mean of each metric by technique (table + grouped bar)
  - Distribution of each metric (violin plots)
  - FCR gate analysis (pass/fail rate per technique)
  - ORR penalty analysis
  - ROA availability (reasoning techniques only)
  - Full results CSV export
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

TECHNIQUES  = ["zero_shot", "cot", "guided_cot", "tot", "got"]
ALL_METRICS = ["fcr", "scs", "wr", "roa", "orr", "wqs"]

TECHNIQUE_COLORS = {
    "zero_shot"  : "#4C72B0",
    "cot"        : "#DD8452",
    "guided_cot" : "#55A868",
    "tot"        : "#C44E52",
    "got"        : "#8172B2",
}
METRIC_COLORS = {
    "fcr" : "#3498db",
    "scs" : "#2ecc71",
    "wr"  : "#9b59b6",
    "roa" : "#e67e22",
    "orr" : "#e74c3c",
    "wqs" : "#1abc9c",
}


def run_metrics_analysis(merged, r1_name, r2_name, out_dir):
    print("\n" + "─"*60)
    print("  [4/4] PER-METRIC BREAKDOWN")
    print("─"*60)

    order     = [t for t in TECHNIQUES if t in merged["technique"].unique()]
    avg_cols  = {m: f"{m}_avg" for m in ALL_METRICS if f"{m}_avg" in merged.columns}

    # ── Mean metrics table ────────────────────────────────────────────────────
    metric_table = (
        merged.groupby("technique")[[v for v in avg_cols.values()]]
        .mean().reindex(order).round(3)
    )
    metric_table.columns = [c.replace("_avg", "").upper()
                             for c in metric_table.columns]
    print("\n  Mean metrics by technique:")
    print(metric_table.to_string())
    metric_table.to_csv(f"{out_dir}/metrics_by_technique.csv")
    print(f"  → Saved: {out_dir}/metrics_by_technique.csv")

    # ── Grouped bar — all metrics per technique ───────────────────────────────
    avail_metrics = list(avg_cols.keys())
    n_metrics = len(avail_metrics)
    x = np.arange(len(order))
    w = 0.8 / n_metrics

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, m in enumerate(avail_metrics):
        vals = merged.groupby("technique")[avg_cols[m]].mean().reindex(order)
        offset = (i - n_metrics/2 + 0.5) * w
        ax.bar(x + offset, vals, width=w, label=m.upper(),
               color=METRIC_COLORS.get(m, "grey"), alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(order)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Mean Score")
    ax.set_title("All Metrics by Technique", fontweight="bold")
    ax.legend(title="Metric", ncol=n_metrics, loc="upper right")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/metrics_grouped_bar.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Saved: {out_dir}/metrics_grouped_bar.png")

    # ── Violin plots ──────────────────────────────────────────────────────────
    n_cols = 3
    n_rows = int(np.ceil(n_metrics / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten()

    for ax, m in zip(axes, avail_metrics):
        col = avg_cols[m]
        tmp = merged[["technique", col]].dropna()
        palette = {t: TECHNIQUE_COLORS.get(t, "grey") for t in order}
        sns.violinplot(data=tmp, x="technique", y=col, order=order,
                       palette=palette, ax=ax, linewidth=1, inner="box")
        ax.set_ylim(-0.05, 1.1)
        ax.set_title(m.upper(), fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("Score")
        ax.tick_params(axis="x", rotation=20)

    for ax in axes[n_metrics:]:
        ax.set_visible(False)

    plt.suptitle("Metric Distributions by Technique", fontsize=14,
                 fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/metrics_violins.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Saved: {out_dir}/metrics_violins.png")

    # ── FCR gate analysis ─────────────────────────────────────────────────────
    if "fcr_avg" in merged.columns:
        fcr_pass = merged.copy()
        fcr_pass["fcr_pass"] = (fcr_pass["fcr_avg"] >= 1.0).astype(int)
        gate = (fcr_pass.groupby("technique")["fcr_pass"]
                .agg(["sum", "count"])
                .reindex(order))
        gate["pass_rate"] = (gate["sum"] / gate["count"]).round(3)
        gate.columns = ["Passed", "Total", "Pass Rate"]
        print("\n  FCR Gate — Pass Rate by Technique:")
        print(gate.to_string())
        gate.to_csv(f"{out_dir}/fcr_gate_analysis.csv")
        print(f"  → Saved: {out_dir}/fcr_gate_analysis.csv")

        fig, ax = plt.subplots(figsize=(8, 4))
        colors = [TECHNIQUE_COLORS.get(t, "grey") for t in order]
        bars = ax.bar(order, gate["Pass Rate"], color=colors,
                      edgecolor="white", alpha=0.9)
        for bar, v in zip(bars, gate["Pass Rate"]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{v:.0%}", ha="center", va="bottom", fontweight="bold")
        ax.set_ylim(0, 1.15)
        ax.axhline(1.0, color="green", linestyle="--", linewidth=1,
                   alpha=0.6, label="100% pass")
        ax.set_ylabel("FCR Pass Rate (FCR = 1.0)")
        ax.set_title("FCR Gate Pass Rate by Technique", fontweight="bold")
        ax.legend()
        plt.tight_layout()
        plt.savefig(f"{out_dir}/fcr_gate.png", dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  → Saved: {out_dir}/fcr_gate.png")

    # ── ORR penalty analysis ──────────────────────────────────────────────────
    if "orr_avg" in merged.columns:
        orr_summary = (merged.groupby("technique")["orr_avg"]
                       .agg(["mean", "std", "max"]).reindex(order).round(3))
        print("\n  ORR Penalty by Technique:")
        print(orr_summary.to_string())

    # ── Side-by-side reviewer comparison for each metric ─────────────────────
    n_metrics_plot = len(avail_metrics)
    fig, axes = plt.subplots(2, int(np.ceil(n_metrics_plot/2)),
                             figsize=(16, 8))
    axes = axes.flatten()

    for ax, m in zip(axes, avail_metrics):
        c1, c2 = f"{m}_{r1_name}", f"{m}_{r2_name}"
        if c1 not in merged.columns or c2 not in merged.columns:
            ax.set_visible(False)
            continue
        r1_means = merged.groupby("technique")[c1].mean().reindex(order)
        r2_means = merged.groupby("technique")[c2].mean().reindex(order)
        x = np.arange(len(order))
        ax.bar(x - 0.18, r1_means, width=0.32, label=r1_name,
               color="#4C72B0", alpha=0.85)
        ax.bar(x + 0.18, r2_means, width=0.32, label=r2_name,
               color="#DD8452", alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(order, rotation=20, fontsize=8)
        ax.set_ylim(0, 1.15)
        ax.set_title(m.upper(), fontweight="bold")
        ax.set_ylabel("Mean")
        if ax == axes[0]:
            ax.legend(title="Reviewer", fontsize=8)

    for ax in axes[n_metrics_plot:]:
        ax.set_visible(False)

    plt.suptitle("Per-Metric Reviewer Comparison", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/metrics_reviewer_comparison.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  → Saved: {out_dir}/metrics_reviewer_comparison.png")

    # ── Full results export ───────────────────────────────────────────────────
    export_cols = (["image_id", "technique", "category"] +
                   [f"{m}_{r1_name}" for m in ALL_METRICS
                    if f"{m}_{r1_name}" in merged.columns] +
                   [f"{m}_{r2_name}" for m in ALL_METRICS
                    if f"{m}_{r2_name}" in merged.columns] +
                   [f"{m}_avg" for m in ALL_METRICS
                    if f"{m}_avg" in merged.columns])
    merged[export_cols].to_csv(f"{out_dir}/full_results.csv", index=False)
    print(f"\n  → Full results saved: {out_dir}/full_results.csv")

In [7]:
"""
CNC Workflow Evaluator — Analysis Runner
Run: python main.py

Runs all analyses in order:
  1. load_data.py       — Load & merge both reviewer files
  2. irr.py             — Inter-rater reliability
  3. technique.py       — Technique comparison
  4. difficulty.py      — Difficulty analysis
  5. metrics.py         — Per-metric breakdown
"""


import os
import warnings
warnings.filterwarnings('ignore')
# from load_data   import load_and_merge
# from irr         import run_irr
# from technique   import run_technique_analysis
# from difficulty  import run_difficulty_analysis
# from metrics     import run_metrics_analysis

# ── Config ────────────────────────────────────────────────────────────────────
FILE_1 = "arya_evaluation_results.json"   # Reviewer 1
FILE_2 = "aaryaman_evaluation_results.json"   # Reviewer 2
OUT_DIR = "output_analysis"
DISAGREE_THRESH = 0.25   # flag if |r1 - r2| >= this

os.makedirs(OUT_DIR, exist_ok=True)

# ── Run ───────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  CNC WORKFLOW EVALUATOR — ANALYSIS")
print("="*60)

merged, r1_name, r2_name = load_and_merge(FILE_1, FILE_2)

run_irr(merged, r1_name, r2_name, OUT_DIR, DISAGREE_THRESH)
run_technique_analysis(merged, r1_name, r2_name, OUT_DIR)
run_difficulty_analysis(merged, OUT_DIR)
run_metrics_analysis(merged, r1_name, r2_name, OUT_DIR)

print(f"\n{'='*60}")
print(f"  ALL DONE — outputs saved to: {OUT_DIR}/")
print(f"{'='*60}\n")


  CNC WORKFLOW EVALUATOR — ANALYSIS

  Reviewer 1 : Arya  (75 records)
  Reviewer 2 : Aaryaman  (75 records)
  Merged     : 75 records
  Techniques : ['cot', 'got', 'guided_cot', 'tot', 'zero_shot']
  Categories : ['hard', 'medium', 'simple']

────────────────────────────────────────────────────────────
  [1/4] INTER-RATER RELIABILITY
────────────────────────────────────────────────────────────

  IRR Summary:
         N  Mean_Arya  Mean_Aaryaman  Mean|diff|  Max|diff|  Cohen_kappa  ICC(2,1)
Metric                                                                            
FCR     75      0.996          0.996       0.000      0.000          NaN     1.000
SCS     72      0.865          0.899       0.049      0.500        0.551     0.889
WR      72      0.722          0.767       0.108      0.500        0.433     0.835
ROA     58      0.841          0.841       0.121      0.500        0.321     0.634
ORR     72      0.060          0.054       0.011      0.200          NaN     0.886
WQS 